In [6]:
# Select compact/full features by walk-forward OOF PR-AUC, tune Elastic Net, and evaluate the frozen model once on test.

In [7]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import (
    project_paths,
    file_hash,
    probability_metrics,
    load_split,
)
from pathlib import Path
import inspect
import json
import platform
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
import statsmodels.api as sm
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from project_config import COMPACT_FEATURE_COLUMNS, PR_AUC_TOLERANCE

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/train_matrix.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
REPORTS = project_paths(ROOT).reports / "05_elastic_net_reporting"
MODELS = project_paths(ROOT).models
C_GRID, L1_RATIOS = [0.001, 0.01, 0.1, 1.0, 10.0], [0.1, 0.5, 0.9]
CLASS_WEIGHTS = {"unweighted": None, "balanced": "balanced"}
MAX_ITER, TOL, SEED = 12000, 1e-4, 42
input_hashes = {}


def make_classifier(C, l1_ratio, class_weight):
    params = dict(
        C=C,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        solver="saga",
        max_iter=MAX_ITER,
        tol=TOL,
        random_state=SEED,
        warm_start=False,
    )
    if (
        inspect.signature(LogisticRegression).parameters["penalty"].default
        != "deprecated"
    ):
        params["penalty"] = "elasticnet"
    return LogisticRegression(**params)


def calibration_fit(labels, probabilities):
    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped))
    try:
        fit = sm.GLM(
            labels,
            sm.add_constant(logits, has_constant="add"),
            family=sm.families.Binomial(),
        ).fit(maxiter=100, disp=0)
        return float(fit.params[0]), float(fit.params[1]), "ok"
    except Exception as exc:
        return np.nan, np.nan, type(exc).__name__


def top_risk(labels, probabilities, fraction):
    count = max(1, int(np.ceil(len(labels) * fraction)))
    indices = np.argsort(-probabilities, kind="stable")[:count]
    cases, total = int(labels[indices].sum()), int(np.asarray(labels).sum())
    return count, cases, cases / total if total else np.nan, cases / count


artifact_path = MODELS / "preprocessing_pipeline.joblib"
input_hashes[artifact_path] = file_hash(artifact_path)
preprocessing = joblib.load(artifact_path)
schema = preprocessing.project_schema_
if schema.get("specification") != "B" or schema.get("fit_split") != "train":
    raise ValueError(
        "Elastic Net requires the training-fitted Spec B preprocessing pipeline"
    )
FEATURES, ID_COLUMNS = (
    schema["matrix_feature_columns"],
    schema["matrix_metadata_columns"],
)
KEYS = ["company_id", "prediction_date"]

In [8]:
# Tune each candidate on every walk-forward training/validation pair.
fold_ids_path = PROCESSED / "walk_forward_fold_ids.csv"
input_hashes[fold_ids_path] = file_hash(fold_ids_path)
manifest_columns = [
    "fold",
    "split",
    "company_id",
    "prediction_date",
    "metadata_event_id",
    "company_group_id",
    "metadata_outcome_end_date",
]
manifest = pd.read_csv(
    fold_ids_path,
    usecols=manifest_columns,
    dtype={
        "company_id": "string",
        "metadata_event_id": "string",
        "company_group_id": "string",
    },
)
manifest["prediction_date"] = pd.to_datetime(manifest.prediction_date, errors="raise")
manifest["metadata_outcome_end_date"] = pd.to_datetime(
    manifest.metadata_outcome_end_date, errors="raise"
)
fold_names = sorted(manifest.loc[manifest.fold.ne("final_test"), "fold"].unique())
if not fold_names:
    raise ValueError("No walk-forward validation folds found")
fold_data = {}
for fold in fold_names:
    members = manifest.loc[manifest.fold.eq(fold)]
    train_ids, val_ids = (
        members.loc[members.split.eq("train")],
        members.loc[members.split.eq("validation")],
    )
    if train_ids.empty or val_ids.empty:
        raise ValueError(f"{fold}: empty training or validation split")
    if set(train_ids.company_group_id).intersection(val_ids.company_group_id):
        raise ValueError(f"{fold}: company group crosses split")
    if not train_ids.metadata_outcome_end_date.max() < val_ids.prediction_date.min():
        raise ValueError(f"{fold}: training outcome reaches validation")
    pipeline_path = (
        MODELS / "walk_forward" / f"{fold}_spec_b_preprocessing_pipeline.joblib"
    )
    train_path = PROCESSED / "walk_forward" / fold / "spec_b" / "train_matrix.csv"
    val_path = PROCESSED / "walk_forward" / fold / "spec_b" / "validation_matrix.csv"
    for path in [pipeline_path, train_path, val_path]:
        input_hashes[path] = file_hash(path)
    fold_pipeline = joblib.load(pipeline_path)
    fold_schema = fold_pipeline.project_schema_
    if (
        fold_schema.get("fit_split") != f"{fold}/train"
        or fold_schema.get("specification") != "B"
    ):
        raise ValueError(f"{fold}: wrong fold pipeline")
    if fold_schema["training_event_ids"] != train_ids.metadata_event_id.tolist():
        raise ValueError(f"{fold}: training IDs differ from pipeline provenance")
    train_frame = pd.read_csv(
        train_path, dtype={"company_id": "string", "metadata_event_id": "string"}
    )
    val_frame = pd.read_csv(
        val_path, dtype={"company_id": "string", "metadata_event_id": "string"}
    )
    fold_features = fold_schema["matrix_feature_columns"]
    if (
        train_frame.columns.tolist() != ID_COLUMNS + fold_features
        or val_frame.columns.tolist() != ID_COLUMNS + fold_features
    ):
        raise ValueError(f"{fold}: matrix schema mismatch")
    if (
        train_frame.metadata_event_id.tolist() != train_ids.metadata_event_id.tolist()
        or val_frame.metadata_event_id.tolist() != val_ids.metadata_event_id.tolist()
    ):
        raise ValueError(f"{fold}: matrix membership/order mismatch")
    fold_data[fold] = (
        train_frame,
        val_frame,
        train_frame.target.to_numpy(dtype=int),
        val_frame.target.to_numpy(dtype=int),
    )

# Choose compact or full features using walk-forward OOF PR-AUC before final tuning.
MATRIX_FEATURES = FEATURES
COMPACT_MATRIX_FEATURES = [
    f"numeric__{column}"
    for column in COMPACT_FEATURE_COLUMNS
    if f"numeric__{column}" in MATRIX_FEATURES
]
CATEGORICAL_MATRIX_FEATURES = [
    column for column in MATRIX_FEATURES if column.startswith("categorical__")
]
COMPACT_MATRIX_FEATURES += CATEGORICAL_MATRIX_FEATURES
FULL_MATRIX_FEATURES = [
    column for column in MATRIX_FEATURES if not column.startswith("missing__")
]
SPECIFICATION_FEATURES = {
    "compact": list(dict.fromkeys(COMPACT_MATRIX_FEATURES)),
    "full": FULL_MATRIX_FEATURES,
}
if not SPECIFICATION_FEATURES["compact"] or not SPECIFICATION_FEATURES["full"]:
    raise ValueError("Compact and full feature specifications must be available")

specification_rows = []
SPECIFICATION_C_GRID = [0.01, 0.1, 1.0]
for specification, spec_features in SPECIFICATION_FEATURES.items():
    for C_value in SPECIFICATION_C_GRID:
        fold_scores, fold_predictions = [], []
        for fold, (train_frame, val_frame, ytr, yva) in fold_data.items():
            fold_spec_features = [
                column for column in spec_features if column in train_frame.columns
            ]
            selector = VarianceThreshold(0.0)
            Xtr = sparse.csr_matrix(
                selector.fit_transform(
                    train_frame[fold_spec_features].to_numpy(dtype=float)
                )
            )
            Xva = sparse.csr_matrix(
                selector.transform(val_frame[fold_spec_features].to_numpy(dtype=float))
            )
            candidate = make_classifier(C_value, 0.9, "balanced")
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                candidate.fit(Xtr, ytr)
            if (
                any(issubclass(w.category, ConvergenceWarning) for w in caught)
                or candidate.n_iter_.max() >= MAX_ITER
            ):
                fold_scores = []
                break
            probability = candidate.predict_proba(Xva)[:, 1]
            fold_scores.append(average_precision_score(yva, probability))
            fold_predictions.append(
                pd.DataFrame(
                    {
                        "metadata_event_id": val_frame.metadata_event_id,
                        "target": yva,
                        "probability": probability,
                    }
                )
            )
        if len(fold_scores) == len(fold_data):
            pooled_predictions = pd.concat(fold_predictions, ignore_index=True)
            specification_rows.append(
                {
                    "specification": specification,
                    "C": C_value,
                    "mean_validation_pr_auc": float(np.mean(fold_scores)),
                    "pooled_oof_pr_auc": average_precision_score(
                        pooled_predictions.target, pooled_predictions.probability
                    ),
                    "predictor_count": len(spec_features),
                }
            )
specification_results = pd.DataFrame(specification_rows)
if specification_results.empty:
    raise RuntimeError("No compact/full specification completed all validation folds")
specification_best = (
    specification_results.sort_values(
        ["specification", "mean_validation_pr_auc", "pooled_oof_pr_auc", "C"],
        ascending=[True, False, False, True],
        kind="stable",
    )
    .drop_duplicates("specification")
    .set_index("specification")
)
compact_pr_auc = float(specification_best.loc["compact", "pooled_oof_pr_auc"])
full_pr_auc = float(specification_best.loc["full", "pooled_oof_pr_auc"])
primary_specification = (
    "compact" if full_pr_auc - compact_pr_auc <= PR_AUC_TOLERANCE else "full"
)
MODEL_FEATURES = SPECIFICATION_FEATURES[primary_specification]
primary_specification_record = {
    "primary_specification": primary_specification,
    "compact_pooled_oof_pr_auc": compact_pr_auc,
    "full_pooled_oof_pr_auc": full_pr_auc,
    "full_minus_compact_pr_auc": full_pr_auc - compact_pr_auc,
    "pr_auc_tolerance": PR_AUC_TOLERANCE,
    "selection_metric": "pooled walk-forward OOF PR-AUC",
    "test_used_for_selection": False,
    "compact_selected_C": float(specification_best.loc["compact", "C"]),
    "full_selected_C": float(specification_best.loc["full", "C"]),
}
MODELS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
(MODELS / "primary_feature_specification.json").write_text(
    json.dumps(primary_specification_record, indent=2), encoding="utf-8"
)
specification_results.to_csv(REPORTS / "feature_specification_selection.csv", index=False)

model_fold_data = {}
for fold, (train_frame, val_frame, ytr, yva) in fold_data.items():
    fold_model_features = [
        column for column in MODEL_FEATURES if column in train_frame.columns
    ]
    selector = VarianceThreshold(0.0)
    Xtr = sparse.csr_matrix(
        selector.fit_transform(train_frame[fold_model_features].to_numpy(dtype=float))
    )
    Xva = sparse.csr_matrix(
        selector.transform(val_frame[fold_model_features].to_numpy(dtype=float))
    )
    model_fold_data[fold] = (train_frame, val_frame, Xtr, Xva, ytr, yva)

fold_rows, candidate_oof = [], {}
for weight_name, class_weight in CLASS_WEIGHTS.items():
    for l1_ratio in L1_RATIOS:
        for C in C_GRID:
            candidate = f"{weight_name}_l1_{l1_ratio:g}_C_{C:g}"
            predictions_by_fold = {}
            for fold, (train_frame, val_frame, Xtr, Xva, ytr, yva) in model_fold_data.items():
                model = make_classifier(C, l1_ratio, class_weight)
                started = time.perf_counter()
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    model.fit(Xtr, ytr)
                warned = any(issubclass(w.category, ConvergenceWarning) for w in caught)
                eligible = bool(
                    not warned
                    and model.n_iter_.max() < MAX_ITER
                    and np.isfinite(model.coef_).all()
                )
                prob = model.predict_proba(Xva)[:, 1]
                fold_rows.append(
                    {
                        "candidate_id": candidate,
                        "fold": fold,
                        "C": C,
                        "l1_ratio": l1_ratio,
                        "class_weight": weight_name,
                        "eligible": eligible,
                        "iterations": int(model.n_iter_.max()),
                        "validation_pr_auc": average_precision_score(yva, prob),
                        "validation_roc_auc": roc_auc_score(yva, prob),
                        "validation_brier_score": brier_score_loss(yva, prob),
                        "elapsed_seconds": time.perf_counter() - started,
                        "warnings": "; ".join(sorted({str(w.message) for w in caught})),
                    }
                )
                if eligible:
                    predictions_by_fold[fold] = pd.DataFrame(
                        {
                            "company_id": val_frame.company_id,
                            "prediction_date": val_frame.prediction_date,
                            "metadata_event_id": val_frame.metadata_event_id,
                            "fold": fold,
                            "target": yva,
                            "bankruptcy_probability": prob,
                        }
                    )
            if len(predictions_by_fold) == len(fold_data):
                candidate_oof[candidate] = predictions_by_fold

fold_tuning = pd.DataFrame(fold_rows)
tuning = fold_tuning.groupby(
    ["candidate_id", "C", "l1_ratio", "class_weight"], as_index=False
).agg(
    eligible=("eligible", "all"),
    folds_completed=("fold", "nunique"),
    mean_validation_pr_auc=("validation_pr_auc", "mean"),
    mean_validation_roc_auc=("validation_roc_auc", "mean"),
    mean_validation_brier_score=("validation_brier_score", "mean"),
    total_iterations=("iterations", "sum"),
)
tuning = (
    tuning.loc[tuning.eligible]
    .sort_values(
        [
            "mean_validation_pr_auc",
            "mean_validation_roc_auc",
            "mean_validation_brier_score",
            "candidate_id",
        ],
        ascending=[False, False, True, True],
        kind="stable",
    )
    .reset_index(drop=True)
)
if tuning.empty:
    raise RuntimeError("No candidate converged across all walk-forward folds")
best = tuning.iloc[0]
oof = pd.concat(list(candidate_oof[best.candidate_id].values()), ignore_index=True)
if oof.metadata_event_id.duplicated().any() or oof.target.nunique() != 2:
    raise ValueError("Pooled OOF predictions must have unique events and both classes")
oof = oof.sort_values(["prediction_date", "company_id"], kind="stable").reset_index(
    drop=True
)

In [9]:
# Freeze the threshold using pooled out-of-fold validation predictions.
y_oof, p_oof = (
    oof.target.to_numpy(dtype=int),
    oof.bankruptcy_probability.to_numpy(dtype=float),
)
thresholds = np.unique(np.r_[0.0, 0.5, p_oof, np.nextafter(p_oof.max(), np.inf)])
threshold_rows = pd.DataFrame(
    [probability_metrics(y_oof, p_oof, t) for t in thresholds]
)
threshold_rows["distance_from_half"] = (threshold_rows.threshold - 0.5).abs()
threshold_rows = threshold_rows.sort_values(
    [
        "balanced_accuracy",
        "sensitivity",
        "specificity",
        "distance_from_half",
        "threshold",
    ],
    ascending=[False, False, False, True, False],
    kind="stable",
).reset_index(drop=True)
OPERATING_THRESHOLD = float(threshold_rows.iloc[0].threshold)

model_configuration = {
    "selected_candidate": best.candidate_id,
    "primary_feature_specification": primary_specification,
    "primary_feature_selection": primary_specification_record,
    "selected_feature_columns": MODEL_FEATURES,
    "C": float(best.C),
    "l1_ratio": float(best.l1_ratio),
    "class_weight": best.class_weight,
    "mean_validation_pr_auc": float(best.mean_validation_pr_auc),
    "walk_forward_folds": fold_names,
    "selection_metric": "mean validation PR-AUC",
    "operating_threshold": OPERATING_THRESHOLD,
    "threshold_source": "pooled out-of-fold validation predictions",
    "threshold_metric": "balanced accuracy with sensitivity/specificity tie-breaks",
    "fit_split": "static training only",
    "solver": "saga",
    "max_iter": MAX_ITER,
    "tol": TOL,
    "seed": SEED,
    "preprocessing_sha256": input_hashes[artifact_path],
    "oof_validation_event_ids": oof.metadata_event_id.tolist(),
}

# Fit the selected specification on the static training split only.
train_path, train_ids_path = PROCESSED / "train_matrix.csv", PROCESSED / "train_ids.csv"
for path in [train_path, train_ids_path]:
    input_hashes[path] = file_hash(path)
train = load_split("train", PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
model_configuration["training_event_ids"] = train.metadata_event_id.tolist()
selector = VarianceThreshold(0.0)
X_train = sparse.csr_matrix(
    selector.fit_transform(train[MODEL_FEATURES].to_numpy(dtype=float))
)
y_train = train.target.to_numpy(dtype=int)
final_model = make_classifier(
    model_configuration["C"],
    model_configuration["l1_ratio"],
    CLASS_WEIGHTS[model_configuration["class_weight"]],
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    final_model.fit(X_train, y_train)
if (
    any(issubclass(w.category, ConvergenceWarning) for w in caught)
    or final_model.n_iter_.max() >= MAX_ITER
):
    raise RuntimeError("Final training fit did not converge")

# Open the test split only after hyperparameters and threshold are fixed.
test = load_split("test", PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
X_test = sparse.csr_matrix(selector.transform(test[MODEL_FEATURES].to_numpy(dtype=float)))
y_test = test.target.to_numpy(dtype=int)
p_test, p_train = (
    final_model.predict_proba(X_test)[:, 1],
    final_model.predict_proba(X_train)[:, 1],
)


def summarize(split, labels, probabilities):
    m = probability_metrics(labels, probabilities, OPERATING_THRESHOLD)
    intercept, slope, status = calibration_fit(labels, probabilities)
    row = {
        "split": split,
        **m,
        "recall": m["sensitivity"],
        "precision": m["tp"] / (m["tp"] + m["fp"]) if m["tp"] + m["fp"] else 0.0,
        "calibration_intercept": intercept,
        "calibration_slope": slope,
        "calibration_status": status,
    }
    for f in [0.01, 0.05, 0.10]:
        n, c, capture, precision = top_risk(labels, probabilities, f)
        tag = f"{int(f * 100)}pct"
        row[f"top_risk_rows_{tag}"], row[f"top_risk_cases_{tag}"] = n, c
        row[f"top_risk_capture_{tag}"], row[f"top_risk_precision_{tag}"] = (
            capture,
            precision,
        )
    return row


metrics = pd.DataFrame(
    [
        summarize("walk_forward_oof_validation", y_oof, p_oof),
        summarize("final_test", y_test, p_test),
    ]
)
train_metrics = summarize("training_diagnostic", y_train, p_train)
oof_predictions = oof.copy()
oof_predictions["predicted_target"] = (p_oof >= OPERATING_THRESHOLD).astype(int)
oof_predictions["prediction_split"] = "walk_forward_oof_validation"
test_predictions = test[ID_COLUMNS].copy()
test_predictions["prediction_probability"] = p_test
test_predictions["predicted_target"] = (p_test >= OPERATING_THRESHOLD).astype(int)
test_predictions["prediction_split"] = "final_test"
predictions = pd.concat(
    [
        oof_predictions.rename(
            columns={"bankruptcy_probability": "prediction_probability"}
        ),
        test_predictions,
    ],
    ignore_index=True,
)

active = selector.get_support()
active_features = np.asarray(MODEL_FEATURES)[active].tolist()
coef = np.zeros(len(MODEL_FEATURES))
coef[active] = final_model.coef_[0]
coefficients = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "included_in_model": active,
        "coefficient": coef,
        "absolute_coefficient": np.abs(coef),
        "nonzero": active & (coef != 0),
    }
)
coefficients = coefficients.sort_values(
    ["absolute_coefficient", "feature"], ascending=[False, True], kind="stable"
)
primary_specification_selector = ColumnTransformer(
    [("selected", "passthrough", [FEATURES.index(c) for c in MODEL_FEATURES])],
    remainder="drop",
)
primary_specification_selector.fit(train[FEATURES].to_numpy(dtype=float))
final_pipeline = Pipeline(
    [
        ("preprocessing", preprocessing),
        ("primary_specification_selector", primary_specification_selector),
        ("training_constant_selector", selector),
        ("classifier", final_model),
    ]
)
model_configuration.update(
    {
        "active_predictors": len(active_features),
        "training_constant_predictors": [
            f for f in MODEL_FEATURES if f not in active_features
        ],
        "training_event_ids": train.metadata_event_id.tolist(),
        "test_event_ids": test.metadata_event_id.tolist(),
        "versions": {
            "python": platform.python_version(),
            "sklearn": sklearn.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "joblib": joblib.__version__,
        },
    }
)
final_pipeline.project_model_ = dict(model_configuration)
final_pipeline.project_model_["matrix_feature_columns"] = MODEL_FEATURES
final_pipeline.project_model_["active_feature_columns"] = active_features
final_pipeline.project_model_["intercept"] = float(final_model.intercept_[0])

In [10]:
# Save the frozen model and audit outputs.
MODELS.mkdir(parents=True, exist_ok=True)
for path in [
    fold_ids_path,
    artifact_path,
    PROCESSED / "train_matrix.csv",
    PROCESSED / "train_ids.csv",
    PROCESSED / "test_matrix.csv",
    PROCESSED / "test_ids.csv",
]:
    if path not in input_hashes:
        input_hashes[path] = file_hash(path)
final_pipeline.project_model_["input_sha256"] = {
    str(path.relative_to(ROOT)): digest for path, digest in input_hashes.items()
}
joblib.dump(final_pipeline, MODELS / "elastic_net_logistic.joblib", compress=3)
configuration_export = {
    k: v
    for k, v in model_configuration.items()
    if k not in {"training_event_ids", "oof_validation_event_ids", "test_event_ids"}
}

# Run the stage analysis after the notebook has saved its model outputs.
import runpy

analysis = runpy.run_path(ROOT / "analysis" / "05_elastic_net_reporting.py")
analysis["generate_report"](globals())